In [ ]:
import numpy as np
from scipy.signal import fftconvolve
from scipy.special import j1
from scipy.ndimage import shift as ndshift
import matplotlib.pyplot as plt

# ── camera sensor ──────────────────────────────────────────────────────────
H, W       = 256, 256   # camera frame [sensor pixels]
FWC        = 1e4        # full-well capacity [electrons]
RN         = 2.0        # readout noise sigma [ADC]

# ── PSF sub-pixel model ────────────────────────────────────────────────────
R_AIRY_PX  = 0.8        # Airy first-dark-ring radius [camera pixels]
PSF_FINE   = 64         # fine-grid size for PSF [fine pixels, square]
OVERSAMPLE = 32         # fine pixels per camera pixel
#   PSF_FINE / OVERSAMPLE = 2×2 camera-pixel footprint per defect

# ── jitter interface (fine pixels) ────────────────────────────────────────
JITTER_MAX_SPX = OVERSAMPLE   # ±1 camera pixel max; reserved for future use

# ── image lag ─────────────────────────────────────────────────────────────
L_KERNEL   = np.array([[3.0, 2.0, 1.0, 0.5, 0.2]])  # fixed absolute ADC tail
LAG_THRESH = 3.0 * RN   # mask threshold: above 3σ noise

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Module 1 — PSF model
#
# A defect is imaged as an Airy disk on a 64×64 fine grid,
# then integrated (binned) 32×32 → 1 camera pixel.
# Result: each defect occupies a 2×2 camera-pixel footprint.
#
# jitter_spx : shift in fine pixels before binning (interface only, default 0)
# ══════════════════════════════════════════════════════════════════════════

def bin2d(img, factor):
    """Sum factor×factor blocks → 1 output pixel."""
    H, W = img.shape
    H2, W2 = H // factor, W // factor
    return (img[:H2*factor, :W2*factor]
            .reshape(H2, factor, W2, factor)
            .sum(axis=(1, 3)))


def make_psf_fine(jitter_spx=0.0):
    """Airy PSF on PSF_FINE×PSF_FINE fine grid, normalised to sum=1."""
    half = PSF_FINE // 2
    y, x = np.mgrid[-half:half, -half:half].astype(float)
    r = np.sqrt(x**2 + y**2)
    u = 3.8317 * r / (R_AIRY_PX * OVERSAMPLE)
    k = np.where(r == 0, 1.0, (2.0 * j1(u) / u)**2)
    k /= k.sum()
    if jitter_spx != 0.0:
        k = ndshift(k, (0.0, jitter_spx), order=3, mode='constant', cval=0.0)
    return k


def make_psf_template(A=1.0, jitter_spx=0.0):
    """
    Camera-pixel PSF: fine grid → bin OVERSAMPLE×OVERSAMPLE.
    Returns shape (PSF_FINE//OVERSAMPLE, PSF_FINE//OVERSAMPLE) = (2, 2).
    Values sum to A.
    """
    return bin2d(make_psf_fine(jitter_spx), OVERSAMPLE) * A


def stamp_psf(frame, x_px, y_px, A=1.0, jitter_spx=0.0):
    """Add a defect PSF centred at (x_px, y_px) in the camera frame."""
    tmpl = make_psf_template(A, jitter_spx)
    th, tw = tmpl.shape
    r0, c0 = y_px - th // 2, x_px - tw // 2
    r1, c1 = r0 + th, c0 + tw
    if r0 >= 0 and c0 >= 0 and r1 <= frame.shape[0] and c1 <= frame.shape[1]:
        frame[r0:r1, c0:c1] += tmpl


# ── sanity check ──────────────────────────────────────────────────────────
_t = make_psf_template(A=100.0)
print(f"Fine grid   : {PSF_FINE}×{PSF_FINE}  OVERSAMPLE={OVERSAMPLE}")
print(f"Camera PSF  : {_t.shape}  sum={_t.sum():.2f} e-")
print(_t)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Module 2 — Saturated image lag
#
# Any pixel above LAG_THRESH is considered illuminated (saturated).
# The illumination mask is convolved with L_KERNEL (1-D x-tail).
# Lag charge is FIXED absolute ADC — independent of defect amplitude A.
# ══════════════════════════════════════════════════════════════════════════

def apply_imagelag(frame, threshold=None, l_kernel=L_KERNEL):
    """
    Returns the lag image (same shape as frame).
    Add to frame externally: obs = frame + apply_imagelag(frame).
    threshold defaults to LAG_THRESH (3σ).
    """
    if threshold is None:
        threshold = LAG_THRESH
    mask = (frame > threshold).astype(float)
    lag  = fftconvolve(mask, l_kernel, mode='full')[:frame.shape[0], :frame.shape[1]]
    return lag

In [ ]:
# ── visualise PSF pipeline ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

k_fine = make_psf_fine()
tmpl   = make_psf_template(A=1.0)

# 1. fine grid
ax = axes[0]
ax.imshow(k_fine, cmap='hot', interpolation='nearest')
ax.set_title(f'Airy PSF  {PSF_FINE}×{PSF_FINE} fine px\n'
             f'r_airy = {R_AIRY_PX} cam-px = {R_AIRY_PX*OVERSAMPLE:.0f} fine px')

# 2. fine grid with binning grid overlay
ax = axes[1]
ax.imshow(k_fine, cmap='hot', interpolation='nearest')
for i in range(0, PSF_FINE + 1, OVERSAMPLE):
    ax.axhline(i - 0.5, color='cyan', lw=0.8)
    ax.axvline(i - 0.5, color='cyan', lw=0.8)
ax.set_title(f'Bin grid: each {OVERSAMPLE}×{OVERSAMPLE} block → 1 camera px')

# 3. binned 2×2 result
ax = axes[2]
im = ax.imshow(tmpl, cmap='hot', interpolation='nearest')
for i in range(tmpl.shape[0]):
    for j in range(tmpl.shape[1]):
        ax.text(j, i, f'{tmpl[i,j]:.4f}', ha='center', va='center',
                fontsize=12, color='black' if tmpl[i,j] > tmpl.max()*0.6 else 'white')
ax.set_title(f'Camera PSF  {tmpl.shape[0]}×{tmpl.shape[1]} px\nsum={tmpl.sum():.4f}')
fig.colorbar(im, ax=ax, shrink=0.7)

fig.suptitle(f'{PSF_FINE}×{PSF_FINE} fine  →  bin {OVERSAMPLE}×{OVERSAMPLE}  →  {PSF_FINE//OVERSAMPLE}×{PSF_FINE//OVERSAMPLE} camera px',
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
def make_noise_frame(H=H, W=W, rn=RN, rng=None):
    if rng is None:
        rng = np.random.RandomState()
    return rng.normal(0.0, rn, size=(H, W))

def plot_frame(frame, title='', rn=RN, defects=None):
    fig, axes = plt.subplots(1, 2, figsize=(16, 3),
                             gridspec_kw={'width_ratios': [4, 1]})

    vmax = max(np.percentile(frame, 99.9), 3*rn)
    ax = axes[0]
    im = ax.imshow(frame, cmap='hot', interpolation='nearest',
                   vmin=-3*rn, vmax=vmax, aspect='auto')
    if defects:
        for (xd, yd, _) in defects:
            ax.plot(xd, yd, 'c+', ms=8, mew=1.2)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label='e-')

    axes[1].hist(frame.ravel(), bins=100, density=True,
                 color='steelblue', alpha=0.7, orientation='horizontal')
    xg = np.linspace(frame.min(), frame.max(), 300)
    axes[1].plot(np.exp(-0.5*(xg/rn)**2) / (rn*np.sqrt(2*np.pi)), xg,
                 'r-', lw=1.5, label=f'N(0,{rn}²)')
    axes[1].set_xlabel('density'); axes[1].legend(fontsize=8); axes[1].grid(True)

    plt.tight_layout()
    plt.show()

# ── test ──────────────────────────────────────────────────────────────────
rng_test = np.random.RandomState(42)
noise = make_noise_frame(rng=rng_test)
plot_frame(noise, title=f'Pure readout noise  N(0, {RN}²)  —  {H}×{W} px')
print(f"sigma = {noise.std():.4f} e-  mean = {noise.mean():.4f} e-")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Demo — PSF + imagelag at A = 20 / 25 / 30 ADC
# Columns: noise only | +PSF | +PSF+lag | lag only
# ══════════════════════════════════════════════════════════════════════════
A_VALS  = [20, 25, 30]
X0, Y0  = W // 2, H // 2   # single defect at frame centre
CROP    = 20                # half-crop around defect for display

def show_pipeline(A, rng):
    noise = make_noise_frame(H, W, RN, rng)

    psf_frame = noise.copy()
    stamp_psf(psf_frame, X0, Y0, A=A)

    lag = apply_imagelag(psf_frame)
    obs = psf_frame + lag

    sl = (slice(Y0-CROP, Y0+CROP+1), slice(X0-CROP, X0+CROP+1))
    vmin, vmax = -3*RN, A * 1.1

    fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
    titles = ['Noise only', f'+PSF  A={A}', '+PSF+lag', 'Lag only']
    imgs   = [noise, psf_frame, obs, lag]

    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(img[sl], cmap='gray', interpolation='nearest',
                  vmin=vmin, vmax=vmax)
        ax.set_title(title, fontsize=10)
        ax.axis('off')

    fig.suptitle(f'A={A} ADC  |  RN={RN} ADC  |  SNR={A/RN:.1f}  |  '
                 f'lag threshold={LAG_THRESH:.1f} ADC  |  crop {2*CROP+1}×{2*CROP+1} px',
                 fontsize=10)
    plt.tight_layout()
    plt.show()

    peak = psf_frame[Y0-1:Y0+1, X0-1:X0+1].max()
    lag_peak = lag[Y0-1:Y0+1, X0+1:X0+6].max()
    print(f'A={A:3d}  PSF peak={peak:.1f} ADC  lag peak={lag_peak:.2f} ADC  '
          f'mask pixels={(psf_frame > LAG_THRESH).sum()}')

rng_demo = np.random.RandomState(7)
for A in A_VALS:
    show_pipeline(A, rng_demo)